In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists retaildb;
use retaildb; 

In [0]:
%sql
create table if not exists retaildb.orders_bronze
(
order_id int,
order_date string,
customer_id int,
order_status string,
filename string,
createdon timestamp
)
using delta
location "gs://databricksfolder/bronze/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_silver
(
order_id int,
order_date date,
customer_id int,
order_status string,
order_year int GENERATED ALWAYS AS (YEAR(order_date)),
order_month int GENERATED ALWAYS AS (MONTH(order_date)),
createdon timestamp,
modifiedon timestamp
)
using delta
location "gs://databricksfolder/silver/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
# 1. Create a 100% empty DataFrame with the exact schema matching your table
empty_silver_df = spark.sql("""
    SELECT 
      CAST(NULL AS INT) AS order_id,
      CAST(NULL AS DATE) AS order_date,
      CAST(NULL AS INT) AS customer_id,
      CAST(NULL AS STRING) AS order_status,
      CAST(NULL AS INT) AS order_year,
      CAST(NULL AS INT) AS order_month,
      CAST(NULL AS TIMESTAMP) AS createdon,
      CAST(NULL AS TIMESTAMP) AS modifiedon
    WHERE 1 = 0
""")

# 2. Write it directly to the GCS path. 
# This forces Delta Lake to physically build the _delta_log directory structure.
empty_silver_df.write \
    .format("delta") \
    .mode("append") \
    .save("gs://databricksfolder/silver/")

print("Physical storage log initialized successfully on GCS!")

In [0]:
%sql
create table if not exists retaildb.orders_gold
(
customer_id int,
order_status string,
order_year int,
num_orders int
)
using delta
location "gs://databricksfolder/gold/"
TBLPROPERTIES(delta.enableChangeDataFeed = true)

In [0]:
# 1. Create a 100% empty DataFrame with the exact schema matching your table
empty_gold_df = spark.sql("""
    SELECT 
      CAST(NULL AS INT) AS customer_id,
      CAST(NULL AS STRING) AS order_status,
      CAST(NULL AS INT) AS order_year,
      CAST(NULL AS INT) AS num_orders
    WHERE 1 = 0
""")

# 2. Write it directly to the GCS path. 
# This forces Delta Lake to physically build the _delta_log directory structure.
empty_gold_df.write \
    .format("delta") \
    .mode("append") \
    .save("gs://databricksfolder/gold/")

print("Physical storage log initialized successfully on GCS!")

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');


In [0]:
%sql
describe history retaildb.orders_silver;

In [0]:
%sql
describe history retaildb.orders_gold;

In [0]:
spark.sql("""
DROP TABLE IF EXISTS retaildb.pipeline_watermarks
""")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==========================================================
# STEP 1: CREATE WATERMARK TABLE (ONE-TIME INITIALIZATION)
# ==========================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS retaildb.pipeline_watermarks (
    pipeline_name STRING,
    last_processed_version BIGINT
)
USING DELTA
""")

# ==========================================================
# STEP 2: GET CURRENT BRONZE VERSION
# ==========================================================

current_version = (
    spark.sql("DESCRIBE HISTORY retaildb.orders_bronze")
         .select("version")
         .first()[0]
)

print(f"Current Bronze Version: {current_version}")

# ==========================================================
# STEP 3: GET LAST PROCESSED VERSION
# ==========================================================

watermark_df = spark.sql("""
SELECT last_processed_version
FROM retaildb.pipeline_watermarks
WHERE pipeline_name = 'bronze_to_silver'
""")

if watermark_df.count() == 0:

    spark.sql("""
    INSERT INTO retaildb.pipeline_watermarks
    VALUES ('bronze_to_silver', -1)
    """)

    last_processed_version = -1

else:

    last_processed_version = watermark_df.first()[0]

print(f"Last Processed Version: {last_processed_version}")

# ==========================================================
# STEP 4: INITIAL LOAD
# ==========================================================

if last_processed_version == -1:

    print("Initial Load Started...")

    changes_df = (
        spark.table("retaildb.orders_bronze")
        .filter("order_id > 0")
        .filter("customer_id > 0")
        .filter("order_status IN ('CLOSED','PENDING_PAYMENT')")
    )

# ==========================================================
# STEP 5: INCREMENTAL LOAD USING CDF
# ==========================================================

else:

    start_version = last_processed_version + 1

    if start_version > current_version:

        print("No New Bronze Versions Found")
        dbutils.notebook.exit("Up To Date")

    print(
        f"Reading CDF from Version "
        f"{start_version} to {current_version}"
    )

    raw_cdf_df = (
        spark.read.format("delta")
        .option("readChangeData", "true")
        .option("startingVersion", start_version)
        .option("endingVersion", current_version)
        .table("retaildb.orders_bronze")
    )

    changes_df = (
        raw_cdf_df
        .filter("_change_type IN ('insert','update_postimage')")
        .filter("order_id > 0")
        .filter("customer_id > 0")
        .filter("order_status IN ('CLOSED','PENDING_PAYMENT')")
    )

    # -----------------------------------------
    # Avoid MERGE conflict:
    #
    # Example:
    # order_id=1 CLOSED
    # order_id=1 PENDING_PAYMENT
    #
    # keep latest CDF record only
    # -----------------------------------------

    window_spec = (
        Window.partitionBy("order_id")
              .orderBy(F.col("_commit_timestamp").desc())
    )

    changes_df = (
        changes_df
        .withColumn(
            "rn",
            F.row_number().over(window_spec)
        )
        .filter("rn = 1")
        .drop("rn")
    )

# ==========================================================
# STEP 6: MERGE INTO SILVER
# ==========================================================

changes_df.createOrReplaceTempView(
    "orders_bronze_changes"
)

spark.sql("""
MERGE INTO retaildb.orders_silver tgt
USING orders_bronze_changes src
ON tgt.order_id = src.order_id

WHEN MATCHED THEN
UPDATE SET
    tgt.customer_id = src.customer_id,
    tgt.order_status = src.order_status,
    tgt.modifiedon = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
INSERT (
    order_id,
    order_date,
    customer_id,
    order_status,
    createdon,
    modifiedon
)
VALUES (
    src.order_id,
    src.order_date,
    src.customer_id,
    src.order_status,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP()
)
""")

print("Silver Merge Completed")

# ==========================================================
# STEP 7: UPDATE WATERMARK
# ==========================================================

spark.sql(f"""
UPDATE retaildb.pipeline_watermarks
SET last_processed_version = {current_version}
WHERE pipeline_name = 'bronze_to_silver'
""")

print(
    f"Watermark Updated -> "
    f"{current_version}"
)

print("Bronze -> Silver Completed Successfully")

In [0]:
# ====================================================================
# 1. Direct Batch Aggregation: Read current Silver table state directly
# ====================================================================
print("Calculating current up-to-date aggregates from Silver layer...")

# This reads the complete, active state of your Silver table files.
# It automatically includes all historical data plus any new file data.
changes_df = spark.sql("""
    SELECT 
        customer_id, 
        order_status, 
        order_year, 
        COUNT(order_id) AS num_orders
    FROM retaildb.orders_silver 
    GROUP BY customer_id, order_status, order_year
""")

# Expose the calculated aggregates as a temporary view for verification if needed
changes_df.createOrReplaceTempView("orders_silver_aggregates")


# ====================================================================
# 2. Run Production Overwrite (Safe for External Tables)
# ====================================================================
print("Executing Gold tier complete table overwrite...")

# By using insertInto with mode("overwrite"), Spark cleanly wipes the old data 
# inside gs://databricksfolder/gold/ and drops in the fresh totals 
# WITHOUT destroying your pre-defined DDL schema or TBLPROPERTIES.
changes_df.write \
    .mode("overwrite") \
    .insertInto("retaildb.orders_gold")

print("Gold tier pipeline processed completely and cleanly via Overwrite!")